# Phase 4 and 5 — Data Cleaning and Text Preprocessing

## Purpose

This notebook is executed only after completing `02_data_understanding.ipynb`.

It applies the cleaning decisions to the frozen raw dataset and exports the
official analysis-ready dataset.

The original raw file is never overwritten.


In [27]:
from pathlib import Path
import sys
import pandas as pd

# Professional project path:
RAW_PATH = Path("../data/raw/combined_reddit_posts.jsonl")
OUTPUT_DIR = Path("../data/processed")
SCRIPT_PATH = Path("../src/preprocessing/prepare_analysis_dataset.py")

# Fallbacks make the uploaded notebook runnable in the current environment.
if not RAW_PATH.exists():
    RAW_PATH = Path("/mnt/data/combined_reddit_posts.jsonl")
if not SCRIPT_PATH.exists():
    SCRIPT_PATH = Path("/mnt/data/prepare_analysis_dataset.py")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SCRIPT_PATH.parent))

# Make both the package root and the module directory importable
sys.path.insert(0, str(SCRIPT_PATH.parent.parent))  # ../src
sys.path.insert(0, str(SCRIPT_PATH.parent))          # ../src/preprocessing

try:
    from preprocessing.prepare_analysis_dataset import prepare_dataset, write_jsonl
except ModuleNotFoundError:
    from prepare_analysis_dataset import prepare_dataset, write_jsonl

print("Raw dataset:", RAW_PATH)
print("Reusable cleaning module:", SCRIPT_PATH)


Raw dataset: ../data/raw/combined_reddit_posts.jsonl
Reusable cleaning module: ../src/preprocessing/prepare_analysis_dataset.py


## 1. Load and normalize

Different sources use different identifiers, date fields, and text fields. The reusable module:

1. coalesces `id`, `post_id`, and `sample_id`;
2. creates a source-qualified `record_id`;
3. generates a deterministic text-hash ID where no source ID exists;
4. standardizes platform and timestamp fields;
5. preserves all source-specific columns.


In [19]:
full_df, ready_df, excluded_df, summary_df = prepare_dataset(RAW_PATH)

display(summary_df)
print("\nRaw shape:", full_df.shape)
print("Analysis-ready shape:", ready_df.shape)


,metric,value
0,raw_rows,5406
1,raw_columns,49
2,invalid_json_lines,0
3,missing_original_id_rows,141
4,exact_text_duplicate_rows,37
5,strict_ai_relevant_rows,2726
6,low_relevance_rows,2680
7,analysis_ready_rows,2666
8,missing_date_rows,141
9,pre_2025_rows,378



Raw shape: (5406, 49)
Analysis-ready shape: (2666, 48)


## 2. Source and platform balance

Source imbalance matters because a model trained or summarized on the full dataset may mostly describe the largest sources rather than developers generally.

In [20]:
platform_profile = (
    full_df.groupby("platform")
    .agg(
        raw_rows=("record_id", "size"),
        ai_relevant_rows=("is_ai_relevant", "sum"),
        analysis_ready_rows=("analysis_eligible", "sum"),
        missing_dates=("date_known", lambda s: (~s).sum()),
    )
    .sort_values("raw_rows", ascending=False)
)
platform_profile["strict_relevance_rate"] = (
    platform_profile["ai_relevant_rows"] / platform_profile["raw_rows"]
)
display(platform_profile)


,raw_rows,ai_relevant_rows,analysis_ready_rows,missing_dates,strict_relevance_rate
platform,,,,,
Reddit,2305,2087,2033,0,0.905423
GitHub Issues,1616,245,245,0,0.151609
Stack Overflow,1108,109,109,0,0.098375
Hacker News,236,146,140,0,0.618644
Other social dataset,141,139,139,141,0.985816


## 3. Why the original relevance rule needs correction

The collectors used substring matching such as:

```python
"ai" in text.lower()
```

This can match unrelated words. The repaired rule uses:

```python
r"(?<![A-Za-z0-9_])ai(?![A-Za-z0-9_])"
```

It also recognizes named tools such as ChatGPT, Claude, Gemini, Copilot, Cursor, DeepSeek, LLM, Llama, Mistral, and Qwen.


In [21]:
source_relevance = (
    full_df.groupby("source")
    .agg(
        rows=("record_id", "size"),
        strict_ai_rows=("is_ai_relevant", "sum"),
    )
)
source_relevance["strict_ai_rate"] = (
    source_relevance["strict_ai_rows"] / source_relevance["rows"]
)
display(source_relevance.sort_values("strict_ai_rate"))


,rows,strict_ai_rows,strict_ai_rate
source,,,
stackoverflow,1108,109,0.098375
github_issues,1616,245,0.151609
arctic_shift,305,170,0.557377
hackernews,236,146,0.618644
huggingface_uit-sentiment-dataset-reddit-2000-balanced,2000,1917,0.958500
huggingface_divde-sentiment_posts,141,139,0.985816


## 4. Duplicate and date checks

- IDs are namespaced by source to prevent cross-platform collisions.
- Exact duplicate text is detected after whitespace and markup normalization.
- Missing dates are retained but excluded from timeline conclusions.
- Records before 2025 are flagged because the research question emphasizes recent discussions.


In [22]:
quality_checks = {
    "Unique record IDs": ready_df["record_id"].is_unique,
    "No blank analysis text": ready_df["text_clean_basic"].str.strip().ne("").all(),
    "No exact-text duplicates": ~ready_df["text_hash"].duplicated().any(),
    "All rows pass strict AI relevance": ready_df["is_ai_relevant"].all(),
    "All rows have at least five words": ready_df["word_count"].ge(5).all(),
}
display(pd.Series(quality_checks, name="passed"))

print("\nDate coverage in analysis-ready data:")
display(
    ready_df[["date_known", "is_recent_2025_plus"]]
    .value_counts(dropna=False)
    .rename("rows")
)


Unique record IDs                    True
No blank analysis text               True
No exact-text duplicates             True
All rows pass strict AI relevance    True
All rows have at least five words    True
Name: passed, dtype: bool


Date coverage in analysis-ready data:


date_known  is_recent_2025_plus
True        True                   2507
False       False                   139
True        False                    20
Name: rows, dtype: int64

## 5. Research-theme coverage

These flags do **not** replace sentiment or emotion models. They are transparent keyword indicators used to understand whether the corpus actually contains enough material about stress, anxiety, burnout, job security, productivity, trust, and privacy.


In [23]:
theme_counts = (
    ready_df["research_themes"]
    .explode()
    .dropna()
    .value_counts()
    .rename_axis("theme")
    .reset_index(name="rows")
)
display(theme_counts)


,theme,rows
0,trust_quality,433
1,job_security,110
2,privacy_security,109
3,productivity,93
4,stress_anxiety,72
5,burnout,3


## 6. Export

Use `analysis_ready_posts.jsonl` for EDA and modeling. Keep `excluded_low_relevance_posts.jsonl` as an audit trail instead of deleting questionable records permanently.


In [24]:
READY_PATH = OUTPUT_DIR / "analysis_ready_posts.jsonl"
EXCLUDED_PATH = OUTPUT_DIR / "excluded_low_relevance_posts.jsonl"
SUMMARY_PATH = OUTPUT_DIR / "data_quality_summary.csv"

write_jsonl(ready_df, READY_PATH)
write_jsonl(excluded_df, EXCLUDED_PATH)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("Saved:", READY_PATH)
print("Saved:", EXCLUDED_PATH)
print("Saved:", SUMMARY_PATH)


Saved: ../data/processed/analysis_ready_posts.jsonl
Saved: ../data/processed/excluded_low_relevance_posts.jsonl
Saved: ../data/processed/data_quality_summary.csv


## Modeling rule


- **VADER:** use `text_clean_basic` because punctuation, capitalization, emojis, and negation carry sentiment.
- **Transformer sentiment model:** use `text_clean_basic`; do not remove stopwords or lemmatize.
- **TF-IDF / LDA:** begin with `text_clean_lexical`, then apply a task-specific tokenizer and stopword list.
- **BERTopic:** use `text_clean_basic` or lightly cleaned raw text because sentence embeddings need natural language context.

